# Comparing the one-dimensional sinc-DVR basis to a finite-difference solution

In [ ]:
import numba
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg

from quantum_systems import BasisSet, GeneralOrbitalSystem, ODQD, ODSincDVR
from configuration_interaction import CISD

In [ ]:
num_dvr = 201
grid = np.linspace(-10, 10, num_dvr)
weight = np.abs(grid[1] - grid[0])
x = np.linspace(-10, 10, 1001)

l = 2
alpha = 1
a = 0.01

In [ ]:
def sinc_basis(weight, point):
    return lambda x, w=weight, x_i=point: np.sinc((x - x_i) / w) / np.sqrt(w)

In [ ]:
spf = [sinc_basis(weight, x_i) for x_i in grid]

In [ ]:
plt.plot(x, spf[50](x))
plt.show()

In [ ]:
class HOPotential:
    def __init__(self, omega=1):
        self.omega = omega

    def __call__(self, x):
        return 0.5 * self.omega**2 * x**2

In [ ]:
def kinetic_energy_mels(weight, i, j):
    if i == j:
        return np.pi**2 / (6 * weight**2)
    return (-1) ** (i - j) / (weight**2 * (i - j) ** 2)

In [ ]:
t = np.zeros((num_dvr, num_dvr))

for i in range(num_dvr):
    for j in range(num_dvr):
        t[i, j] = kinetic_energy_mels(weight, i, j)

In [ ]:
np.testing.assert_allclose(t, t.T)

In [ ]:
potential = HOPotential()

In [ ]:
v = potential(grid)

In [ ]:
h = t + np.diag(v)

In [ ]:
eps, C = scipy.linalg.eigh(h)

In [ ]:
eps[:10]

In [ ]:
fig, ax = plt.subplots()

ax.plot(x, potential(x))

for i in range(10):
    ax.plot(grid, np.abs(C[:, i] / np.sqrt(weight)) ** 2 + eps[i])


ax.set_xlim(-5, 5)
ax.set_ylim(0, 10)
ax.grid()

In [ ]:
def shielded_coulomb_operator(x_1, x_2, kappa, a):
    return kappa / np.sqrt((x_1 - x_2) ** 2 + a**2)

In [ ]:
def coulomb_mels(grid, kappa, a):
    num_dvr = len(grid)

    u = np.zeros((num_dvr, num_dvr))

    for p in range(num_dvr):
        u[p, :] = shielded_coulomb_operator(grid[p], grid, kappa, a)

    return u

In [ ]:
u_pq = coulomb_mels(grid, (alpha := 1), (a := 0.01))

In [ ]:
u_pq_b = shielded_coulomb_operator(grid[None, :], grid[:, None], alpha, a)

In [ ]:
u_pq_b.shape

In [ ]:
np.testing.assert_allclose(u_pq, u_pq_b)

In [ ]:
C_new = C[:, :l]

In [ ]:
C_new.shape

In [ ]:
h_new = np.einsum("pa, pq, qb -> ab", C_new, h, C_new)

In [ ]:
np.diag(h_new)

In [ ]:
u_new = np.einsum(
    "pa, qb, pc, qd, pq -> abcd", C_new, C_new, C_new, C_new, u_pq, optimize=True
)

In [ ]:
system = GeneralOrbitalSystem(
    2, (odqd := ODQD(l, 10, 201, a=0.01, alpha=alpha, potential=potential)).copy_basis()
)
cisd_fd = CISD(system, verbose=True).compute_ground_state()
print(cisd_fd.energies[:10])

In [ ]:
bs = BasisSet(l, dim=1)
bs.h = h_new
bs.u = u_new
bs.s = np.eye(l)

system_dvr = GeneralOrbitalSystem(2, bs)
cisd_dvr = CISD(system_dvr, verbose=True).compute_ground_state()
print(cisd_dvr.energies[:10])

I am unsure as to which method is best. As far as I know, the finite-difference basis is not variational, and that is perhaps best seen in the single-particle eigenenergies. We now the exact solution for the harmonic oscillator system, and the finite-difference basis undershoots these values. I would assume that the DVR-basis is in fact better, but the two-body energy is generally higher than in the finite-difference case. Looking at the one-body eigenenergies again we see that the DVR basis hits the exact value with very few grid points. This makes it much more appealing.

In [ ]:
odsdvr = ODSincDVR(num_dvr, grid[-1], potential=potential, a=a, alpha=alpha)

In [ ]:
np.testing.assert_allclose(odsdvr.u, u_pq)
np.testing.assert_allclose(odsdvr.h, h)

In [ ]:
eps_o, C_o = scipy.linalg.eigh(odsdvr.h)

In [ ]:
C_o_new = C_o[:, :l]

In [ ]:
np.testing.assert_allclose(np.abs(C_o_new), np.abs(C_new), atol=1e-12)

In [ ]:
u_o_new = odsdvr.transform_two_body_elements(odsdvr.u, C_o_new, np)

In [ ]:
np.testing.assert_allclose(np.abs(u_new), np.abs(u_o_new), atol=1e-12)

In [ ]:
@numba.njit
def _shielded_coulomb(x_1, x_2, alpha, a):
    return alpha / np.sqrt((x_1 - x_2) ** 2 + a**2)


@numba.njit
def _construct_inner_shielded_coulomb_integral(spf_2, grid_1, grid_2, alpha, a):
    num_grid_1_points = len(grid_1)
    num_grid_2_points = len(grid_2)
    l = len(spf_2)
    inner_integral = np.zeros((l, l, num_grid_1_points))

    for i in range(num_grid_1_points):
        coulomb = _shielded_coulomb(grid_1[i], grid_2, alpha, a)
        for q in range(l):
            for s in range(l):
                inner_integral[q, s, i] = np.trapz(
                    spf_2[q] * coulomb * spf_2[s],
                    grid_2,
                )

    return inner_integral


@numba.njit
def construct_shielded_coulomb_interaction_matrix_elements_dist(
    spf_1, spf_2, grid_1, grid_2, alpha, a
):
    l_1 = len(spf_1)
    l_2 = len(spf_2)
    inner_integral = _construct_inner_shielded_coulomb_integral(
        spf_2, grid_1, grid_2, alpha, a
    )
    u = np.zeros((l_1, l_2, l_1, l_2))

    for p in range(l_1):
        for q in range(l_2):
            for r in range(l_1):
                for s in range(l_2):
                    u[p, q, r, s] = np.trapz(
                        spf_1[p] * inner_integral[q, s] * spf_1[r],
                        grid_1,
                    )

    return u

In [ ]:
u_dvr_fd = construct_shielded_coulomb_interaction_matrix_elements_dist(
    C_new.T / np.sqrt(weight), C_new.T / np.sqrt(weight), grid, grid, alpha, a
)

In [ ]:
np.testing.assert_allclose(np.abs(u_dvr_fd), np.abs(u_new), atol=1e-12)

In [ ]:
for p in range(l):
    for q in range(l):
        for r in range(l):
            for s in range(l):
                print(u_dvr_fd[p, q, r, s], u_new[p, q, r, s])

In [ ]:
for p in range(l):
    for q in range(l):
        for r in range(l):
            for s in range(l):
                print((p, q, r, s), odqd.u[p, q, r, s].real, u_new[p, q, r, s])

In [ ]:
u_lol = construct_shielded_coulomb_interaction_matrix_elements_dist(
    odqd.spf[:1].real, odqd.spf[:1].real, odqd.grid, odqd.grid, alpha, a
)

In [ ]:
u_lol

In [ ]:
np.testing.assert_allclose(np.abs(u_new), np.abs(odqd.u), atol=1e-3)

In [ ]:
np.abs(u_new.ravel())

In [ ]:
np.abs(odqd.u.real.ravel())

In [ ]:
np.testing.assert_allclose(np.abs(u_dvr_fd), np.abs(odqd.u), atol=1e-3)

In short there seems to be a difference for Coulomb interaction matrix elements when using DVR and FD. Computing the matrix elements using either DVR directly and then transforming to the eigenbasis of $\hat{h}$, or transforming the basis and then computing the Coulomb matrix elements using the same numerical integration scheme as for the FD-basis gives the same results.

However, FD and DVR differ. In particular, it seems that the number of grid points greatly changes the values for the Coulomb elements. For the same number of grid points they are comparable.